# Day 13: Benchmark LLM Parameter Variance

Welcome to Day 13! Today we dive deep into the knobs and dials that control LLM output behavior: **Temperature**, **Top-P (Nucleus Sampling)**, and **Frequency Penalty**.

## Core Theory (Just-in-Time)

**Why alter these parameters?**
As an AI Engineer, you don't just want text; you want the *right kind* of text. A code generation task requires highly deterministic output, while a creative brainstorming agent requires diverse and surprising ideas.

1. **Temperature:** Controls the "randomness" of the model's predictions. 
   - Technically, it scales the logits before the softmax function is applied.
   - **Low (e.g., 0.0 - 0.3):** Makes the model "greedy", consistently picking the highest-probability token. Best for factual Q&A or code.
   - **High (e.g., 0.7 - 1.5):** Flattens the probability distribution, allowing lower-probability tokens to be selected. Increases creativity but also hallucinations.
2. **Top-P (Nucleus Sampling):** An alternative to temperature. Instead of altering the probabilities, it restricts the sampling pool to the smallest set of tokens whose cumulative probability exceeds `p`.
   - **p=0.9:** The model only considers the tokens making up the top 90% of the probability mass. This dynamically cuts off the "long tail" of highly improbable tokens.
3. **Frequency Penalty:** Penalizes new tokens based on their existing frequency in the generated text so far.
   - Useful for preventing the model from looping or repeating the same phrases. Positive values decrease the likelihood of repetition.


## Code Implementation

We will use `langchain_openai.ChatOpenAI` to benchmark the variance across 10 iterations of the same prompt, explicitly changing these parameters.
Notice our strict adherence to type hinting (`pydantic`) and logging, reflecting production-grade AI engineering.

In [ ]:
import os
import logging
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ExperimentResult(BaseModel):
    iteration: int
    temperature: float
    top_p: float
    frequency_penalty: float
    response: str

def run_parameter_experiment(prompt: str, iterations: int = 10) -> List[ExperimentResult]:
    """
    Runs 10 iterations of the same prompt while varying Temperature, Top-P, and Frequency Penalty.
    Returns a strictly typed list of ExperimentResult objects.
    """
    results: List[ExperimentResult] = []
    
    # 10 distinct configurations to demonstrate variance
    configs = [
        {"temp": 0.0, "top_p": 1.0, "freq": 0.0}, # Baseline Greedy
        {"temp": 0.5, "top_p": 1.0, "freq": 0.0},
        {"temp": 1.0, "top_p": 1.0, "freq": 0.0}, # Standard Default
        {"temp": 1.5, "top_p": 1.0, "freq": 0.0}, # High Randomness
        {"temp": 2.0, "top_p": 1.0, "freq": 0.0}, # Max Randomness (Often Gibberish)
        {"temp": 1.0, "top_p": 0.5, "freq": 0.0}, # Restricted Nucleus
        {"temp": 1.0, "top_p": 0.1, "freq": 0.0}, # Highly Restricted Nucleus
        {"temp": 1.0, "top_p": 1.0, "freq": 1.0}, # Moderate Frequency Penalty
        {"temp": 1.0, "top_p": 1.0, "freq": 2.0}, # Max Frequency Penalty
        {"temp": 1.5, "top_p": 0.5, "freq": 1.0}, # Hybrid: High Temp, Constrained Nucleus, Penalized Repetition
    ]
    
    for i in range(min(iterations, len(configs))):
        cfg = configs[i]
        logger.info(f"Iteration {i+1}: Temp={cfg['temp']}, Top-P={cfg['top_p']}, FreqPen={cfg['freq']}")
        
        try:
            # Langchain's ChatOpenAI allows passing these via model_kwargs
            llm = ChatOpenAI(
                model="gpt-3.5-turbo",
                temperature=cfg['temp'],
                model_kwargs={
                    "top_p": cfg['top_p'],
                    "frequency_penalty": cfg['freq']
                }
            )
            
            messages = [HumanMessage(content=prompt)]
            response = llm.invoke(messages)
            
            result = ExperimentResult(
                iteration=i + 1,
                temperature=cfg['temp'],
                top_p=cfg['top_p'],
                frequency_penalty=cfg['freq'],
                response=str(response.content)
            )
            results.append(result)
            
        except Exception as e:
            logger.error(f"Error during iteration {i+1}: {e}")
            
    return results

if __name__ == "__main__":
    if not os.environ.get("OPENAI_API_KEY"):
        logger.warning("OPENAI_API_KEY not found. Skipping execution.")
    else:
        test_prompt = "Write a one-sentence creative description of a futuristic city."
        experiment_data = run_parameter_experiment(prompt=test_prompt, iterations=10)
        
        print("\n--- Experiment Results ---")
        for res in experiment_data:
            print(f"Iter {res.iteration:02d} | T:{res.temperature:.1f} P:{res.top_p:.1f} F:{res.frequency_penalty:.1f} | {res.response}")


## Common Pitfalls in Production

1. **Altering Temperature and Top-P Simultaneously:** It is a well-known best practice to alter *either* temperature or top-p, but generally not both at the same time. If you do both, it becomes impossible to isolate which parameter caused a specific change in the output distribution.
2. **High Temperature Hallucinations:** Setting temperature too high (e.g., `> 1.5`) without constraining the output format will rapidly degrade text into absolute gibberish or severe hallucinations, as the model starts favoring highly improbable tokens.
3. **Over-Penalizing Frequency:** Setting `frequency_penalty` too high (e.g., near `2.0`) can force the model to avoid common and grammatically necessary words (like "the", "and", "is"), resulting in jarring, unreadable prose.

## Practical Lab / Homework

**Task:** Build an actionable `ParameterMatrixRunner` that takes a list of configurations, queries the LLM, and calculates a basic "Lexical Diversity Score" (Unique Words / Total Words). 

This is fully implemented below. Review the `RunMetrics` Pydantic model and the extraction of diversity metrics as a proxy for evaluating parameter variance.

In [ ]:
import os
import re
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

class LLMConfig(BaseModel):
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    top_p: float = Field(default=1.0, ge=0.0, le=1.0)
    frequency_penalty: float = Field(default=0.0, ge=-2.0, le=2.0)

class RunMetrics(BaseModel):
    config: LLMConfig
    response_text: str
    word_count: int
    unique_word_count: int
    lexical_diversity_score: float

class ParameterMatrixRunner:
    """
    Executes a prompt across a matrix of LLM configurations and calculates diversity metrics.
    """
    def __init__(self, model_name: str = "gpt-3.5-turbo"):
        self.model_name = model_name

    def calculate_metrics(self, config: LLMConfig, text: str) -> RunMetrics:
        # Simple tokenization by word
        words = re.findall(r'\b\w+\b', text.lower())
        word_count = len(words)
        unique_words = len(set(words))
        diversity_score = unique_words / word_count if word_count > 0 else 0.0
        
        return RunMetrics(
            config=config,
            response_text=text,
            word_count=word_count,
            unique_word_count=unique_words,
            lexical_diversity_score=diversity_score
        )

    def run_matrix(self, prompt: str, configs: List[LLMConfig]) -> List[RunMetrics]:
        results: List[RunMetrics] = []
        
        for idx, config in enumerate(configs):
            print(f"Running config {idx + 1}/{len(configs)}: {config}")
            try:
                llm = ChatOpenAI(
                    model=self.model_name,
                    temperature=config.temperature,
                    model_kwargs={
                        "top_p": config.top_p,
                        "frequency_penalty": config.frequency_penalty
                    }
                )
                
                messages = [HumanMessage(content=prompt)]
                response = llm.invoke(messages)
                response_text = str(response.content)
                
                metrics = self.calculate_metrics(config, response_text)
                results.append(metrics)
                
            except Exception as e:
                print(f"Error with config {config}: {e}")
                
        return results

# --- Lab Execution ---
if __name__ == "__main__":
    if not os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found. Skipping Lab Execution.")
    else:
        lab_prompt = "Explain the significance of the Turing Test in modern AI in one paragraph."
        
        test_configs = [
            LLMConfig(temperature=0.0, top_p=1.0, frequency_penalty=0.0), # Baseline Greedy
            LLMConfig(temperature=1.0, top_p=1.0, frequency_penalty=0.0), # Standard
            LLMConfig(temperature=1.5, top_p=0.5, frequency_penalty=0.0), # High Temp, Constrained P
            LLMConfig(temperature=1.0, top_p=1.0, frequency_penalty=1.5), # High Penalty
        ]
        
        runner = ParameterMatrixRunner()
        lab_results = runner.run_matrix(lab_prompt, test_configs)
        
        print("\n=== Lab Results ===")
        for res in lab_results:
            print(f"\nConfig: T={res.config.temperature}, P={res.config.top_p}, FreqPen={res.config.frequency_penalty}")
            print(f"Diversity Score: {res.lexical_diversity_score:.2f} ({res.unique_word_count}/{res.word_count} unique words)")
            print(f"Response Preview: {res.response_text[:100]}...")
